In [291]:
import json
import re
import sqlite3
from pathlib import Path
from typing import Any
from unittest.mock import DEFAULT

import pandas as pd


In [292]:
DEFAULT_PATH = "/mnt/c/Users/vanes/repos/00_BFH/08_semester/00_BachelorThesis/langgraphandopenwebui/spider/"

FILE_NAME = "spider_test_with_ask_responses_offset_0_max_rows_100"

DATA_FILE_PATH = DEFAULT_PATH + "notebooks/result/" + FILE_NAME + ".json"
OUTPUT_FILE_PATH = DEFAULT_PATH + "notebooks/evaluation/" + FILE_NAME + ".json"

In [293]:
def extract_sql(answer: Any) -> str | None:
    if answer is None or not isinstance(answer, str):
        return None

    text = answer.strip()

    if not text:
        return None

    # Prefer fenced SQL blocks
    fenced_match = re.search(
        r"```(?:sqlite)?\s*(.*?)```",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    if fenced_match:
        return fenced_match.group(1).strip().rstrip(";")

    # If the response contains prose, try to extract from first SELECT/WITH
    sql_start = re.search(r"\b(SELECT|WITH)\b", text, flags=re.IGNORECASE)
    if sql_start:
        sql = text[sql_start.start():].strip()
        return sql.rstrip(";")

    return None

In [294]:
def normalize_value(value: Any, float_precision: int = 6) -> Any:
    """
    Normalize SQLite values for fair comparison.
    """
    if isinstance(value, float):
        return round(value, float_precision)

    if isinstance(value, bytes):
        return value.decode("utf-8", errors="ignore")

    return value


def normalize_rows(rows: list[tuple], float_precision: int = 6) -> list[tuple]:
    """
    Normalize every value in every row.
    """
    return [
        tuple(normalize_value(value, float_precision) for value in row)
        for row in rows
    ]

In [295]:
def execute_sqlite_query(db_path: str | Path, query: str) -> list[tuple]:
    """
    Execute a single SQLite query and return rows.
    """
    db_path = Path(db_path)

    if not db_path.exists():
        raise FileNotFoundError(f"Database not found: {db_path}")

    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute(query)
        rows = cursor.fetchall()

    return rows

In [296]:
def sort_rows_safely(rows: list[tuple]) -> list[tuple]:
    return sorted(rows, key=lambda row: tuple(repr(value) for value in row))

In [297]:
def evaluate_generated_query(
    db_path: str | Path,
    generated_query: str | None,
    golden_query: str,
    *,
    ordered: bool = False,
    float_precision: int = 6,
) -> dict:
    """
    Compare generated SQL result against golden SQL result.

    Returns a dictionary with correctness, errors, and row counts.
    """

    if not generated_query:
        return {
            "correct": False,
            "error": "No generated SQL found",
            "generated_sql": generated_query,
            "golden_sql": golden_query,
            "generated_result": None,
            "golden_result": None,
            "generated_row_count": None,
            "golden_row_count": None,
        }

    try:
        golden_rows = execute_sqlite_query(db_path, golden_query)
    except Exception as error:
        return {
            "correct": False,
            "error": f"Golden query failed: {error}",
            "generated_sql": generated_query,
            "golden_sql": golden_query,
            "generated_result": None,
            "golden_result": None,
            "generated_row_count": None,
            "golden_row_count": None,
        }

    try:
        generated_rows = execute_sqlite_query(db_path, generated_query)
    except Exception as error:
        return {
            "correct": False,
            "error": f"Generated query failed: {error}",
            "generated_sql": generated_query,
            "golden_sql": golden_query,
            "generated_result": None,
            "golden_result": normalize_rows(golden_rows, float_precision),
            "generated_row_count": None,
            "golden_row_count": len(golden_rows),
        }

    golden_result = normalize_rows(golden_rows, float_precision)
    generated_result = normalize_rows(generated_rows, float_precision)

    if ordered:
        correct = generated_result == golden_result
    else:
        correct = sort_rows_safely(generated_result) == sort_rows_safely(golden_result)

    return {
        "correct": correct,
        "error": None,
        "generated_sql": generated_query,
        "golden_sql": golden_query,
        "generated_result": generated_result,
        "golden_result": golden_result,
        "generated_row_count": len(generated_result),
        "golden_row_count": len(golden_result),
    }

In [298]:
def get_db_path(db_id: str, database_root: str | Path) -> Path:
    database_root = Path(database_root)
    return database_root / db_id / f"{db_id}.sqlite"

In [299]:
def evaluate_row(
    row: pd.Series,
    database_root: str | Path,
    response_column: str,
    *,
    ordered: bool = False,
) -> dict:
    """
    Evaluate either thinking or non-thinking response for a dataframe row.
    """

    db_id = row["db_id"]
    db_path = get_db_path(db_id, database_root)

    golden_query = row["query"]

    response_obj = row.get(response_column, {})

    if not isinstance(response_obj, dict):
        generated_answer = None
    else:
        generated_answer = response_obj.get("ask_answer")

        # fallback if ask_answer is missing
        if generated_answer is None:
            generated_answer = (
                response_obj
                .get("response_json", {})
                .get("answer")
            )

    generated_sql = extract_sql(generated_answer)

    return evaluate_generated_query(
        db_path=db_path,
        generated_query=generated_sql,
        golden_query=golden_query,
        ordered=ordered,
    )

In [300]:
def add_sql_evaluation_columns(
    df: pd.DataFrame,
    database_root: str | Path,
    *,
    ordered: bool = False,
) -> pd.DataFrame:
    df = df.copy()

    df["thinking_eval"] = df.apply(
        lambda row: evaluate_row(
            row,
            database_root,
            "ask_response_thinking",
            ordered=ordered,
        ),
        axis=1,
    )

    df["non_thinking_eval"] = df.apply(
        lambda row: evaluate_row(
            row,
            database_root,
            "ask_response_non_thinking",
            ordered=ordered,
        ),
        axis=1,
    )

    df["thinking_correct"] = df["thinking_eval"].apply(lambda x: x["correct"])
    df["non_thinking_correct"] = df["non_thinking_eval"].apply(lambda x: x["correct"])

    df["thinking_error"] = df["thinking_eval"].apply(lambda x: x["error"])
    df["non_thinking_error"] = df["non_thinking_eval"].apply(lambda x: x["error"])

    df["thinking_generated_sql"] = df["thinking_eval"].apply(lambda x: x["generated_sql"])
    df["non_thinking_generated_sql"] = df["non_thinking_eval"].apply(lambda x: x["generated_sql"])

    df["thinking_generated_row_count"] = df["thinking_eval"].apply(
        lambda x: x["generated_row_count"]
    )
    df["non_thinking_generated_row_count"] = df["non_thinking_eval"].apply(
        lambda x: x["generated_row_count"]
    )

    df["golden_row_count"] = df["thinking_eval"].apply(
        lambda x: x["golden_row_count"]
    )

    return df

In [301]:
with open(DATA_FILE_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

project_root = Path(
    r"/spider"
)

database_root = Path(
    "/mnt/c/Users/vanes/repos/00_BFH/08_semester/00_BachelorThesis/langgraphandopenwebui/spider/spider_data/database"
)

df = add_sql_evaluation_columns(
    df,
    database_root=database_root,
    ordered=False,
)

df[
    [
        "db_id",
        "question",
        "thinking_correct",
        "non_thinking_correct",
        "thinking_error",
        "non_thinking_error",
        "thinking_generated_sql",
        "non_thinking_generated_sql",
    ]
]

,db_id,question,thinking_correct,non_thinking_correct,thinking_error,non_thinking_error,thinking_generated_sql,non_thinking_generated_sql
0,concert_singer,How many singers do we have?,True,True,None,None,SELECT COUNT(singer.singer_id) AS total_singer...,SELECT COUNT(*) FROM singer
1,concert_singer,What is the total number of singers?,True,True,None,None,SELECT COUNT(singer.singer_id) AS total_singer...,SELECT count(*) FROM singer
2,concert_singer,"Show name, country, age for all singers ordere...",True,True,None,None,"SELECT name, country, age FROM singer ORDER BY...","SELECT\n name,\n country,\n age\nFROM singe..."
3,concert_singer,"What are the names, countries, and ages for ev...",True,True,None,None,"SELECT name, country, age FROM singer ORDER BY...","SELECT\n name,\n country,\n age\nFROM singe..."
4,concert_singer,"What is the average, minimum, and maximum age ...",True,False,None,"Generated query failed: near ""sql"": syntax error","SELECT AVG(age) AS average_age, MIN(age) AS mi...","sql\nSELECT\n AVG(age),\n MIN(age),\n MAX(a..."
...,...,...,...,...,...,...,...,...
95,car_1,Which model of the car has the minimum horsepo...,False,False,None,No generated SQL found,SELECT T2.model FROM cars_data AS T1 INNER JOI...,None
96,car_1,What is the model of the car with the smallest...,False,False,None,No generated SQL found,SELECT T2.model FROM cars_data AS T1 INNER JOI...,None
97,car_1,Find the model of the car whose weight is belo...,True,True,None,None,SELECT T1.model FROM car_names AS T1 INNER JOI...,SELECT\n T1.model\nFROM car_names AS T1\nINNE...
98,car_1,What is the model for the car with a weight sm...,True,False,None,No generated SQL found,SELECT T1.model FROM car_names AS T1 INNER JOI...,None


In [302]:
thinking_accuracy = df["thinking_correct"].mean()
non_thinking_accuracy = df["non_thinking_correct"].mean()

print(f"Thinking accuracy:     {thinking_accuracy:.2%}")
print(f"Non-thinking accuracy: {non_thinking_accuracy:.2%}")

Thinking accuracy:     68.00%
Non-thinking accuracy: 62.00%


In [303]:
df.to_json(
    OUTPUT_FILE_PATH,
    orient="records",
    indent=2,
    force_ascii=False,
)